### 1. Write a function to perform image preprocessing for inference(Pytorch)

In [43]:
from PIL import Image
from torchvision import transforms

def preprocess_image(image_path):
  

    # Define preprocessing pipeline
    transform = transforms.Compose([
        transforms.Resize((32, 32)),
        transforms.ToTensor(),
         transforms.Normalize(
        (0.5,0.5,0.5),
        (0.5,0.5,0.5)
         )
    ])

    # Load image
    image = Image.open(image_path).convert("RGB")

    # Apply transformations
    image = transform(image)

    # Add batch dimension
    image = image.unsqueeze(0)

    return image

In [44]:
input_image = preprocess_image("Images_to_use/flower.jpg")


In [45]:
#Image is preprocessed as mentioned in the function
input_image

tensor([[[[-0.8745, -0.8431, -0.8667,  ..., -1.0000, -1.0000, -1.0000],
          [-0.8667, -0.8275, -0.8745,  ..., -0.9922, -1.0000, -1.0000],
          [-0.8745, -0.8431, -0.9059,  ..., -0.9765, -1.0000, -1.0000],
          ...,
          [-0.8980, -0.8902, -0.8980,  ..., -0.8980, -0.8745, -0.9059],
          [-0.9216, -0.9137, -0.8980,  ..., -0.9294, -0.9294, -0.9059],
          [-0.9216, -0.9059, -0.9059,  ..., -0.9294, -0.9294, -0.9137]],

         [[-0.6863, -0.5922, -0.6627,  ..., -1.0000, -1.0000, -1.0000],
          [-0.6784, -0.5922, -0.7020,  ..., -0.9922, -1.0000, -1.0000],
          [-0.6863, -0.6392, -0.8118,  ..., -0.9765, -1.0000, -1.0000],
          ...,
          [-0.8588, -0.8431, -0.8431,  ..., -0.7490, -0.7804, -0.8275],
          [-0.8745, -0.8431, -0.8275,  ..., -0.8196, -0.8588, -0.8510],
          [-0.8745, -0.8353, -0.8196,  ..., -0.8667, -0.8745, -0.8667]],

         [[-0.9059, -0.8667, -0.8824,  ..., -1.0000, -1.0000, -1.0000],
          [-0.8902, -0.8667, -

In [47]:
import torch

model = CNN()
model.load_state_dict(torch.load("cnn_cifar10.pth"))
model.eval()

input_image = preprocess_image("Images_to_use/tiger.jpg")

with torch.no_grad():
    output = model(input_image)
    predicted_class = torch.argmax(output, dim=1)

print("Predicted Class:", predicted_class.item())

Predicted Class: 6


## 3: Build a CNN to classify CIFAR-10 images (PyTorch)
### Problem: Create a CNN model that classifies images from the CIFAR-10 dataset with accuracy above 60%.


In [19]:
# importing the libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader


In [20]:
# image Preprocessing
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),
                         (0.5,0.5,0.5))
])


In [21]:
#loading the dataset
train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [33]:
#Define the model
class CNN(nn.Module):

    def __init__(self):
        super(CNN, self).__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*4*4,512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512,10)
        )

    def forward(self,x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = CNN()

In [34]:
# Define loss and optimizer
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(
    model.parameters(),
    lr=0.001
)

In [35]:
# Train the model
epochs = 20 
for epoch in range(epochs):
    model.train()
    running_loss=0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs=model(images)
        loss = criterion(outputs,labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() 
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")

Epoch 1, Loss: 1.4539
Epoch 2, Loss: 1.0355
Epoch 3, Loss: 0.8483
Epoch 4, Loss: 0.7225
Epoch 5, Loss: 0.6325
Epoch 6, Loss: 0.5585
Epoch 7, Loss: 0.4969
Epoch 8, Loss: 0.4369
Epoch 9, Loss: 0.3900
Epoch 10, Loss: 0.3473
Epoch 11, Loss: 0.3107
Epoch 12, Loss: 0.2804
Epoch 13, Loss: 0.2572
Epoch 14, Loss: 0.2347
Epoch 15, Loss: 0.2141
Epoch 16, Loss: 0.2011
Epoch 17, Loss: 0.1921
Epoch 18, Loss: 0.1842
Epoch 19, Loss: 0.1737
Epoch 20, Loss: 0.1657


In [36]:
# Evaluating the model
# Correct is nothing but correct predictions during testing phase, total is nothing but total number of images used for testing
model.eval()
correct = 0
total = 0
# we don't need to calculate gradients as there is no need for updating weights in testing phase
with torch.no_grad():
    for images, labels in test_loader:
        outputs=model(images)
        # After passing the images we will get scores for each class and we will take the maximum score as the predicted class
        # we get both score and label after passing the image, here predicted class label is stored in predicteds
        # here 1 indicates we see values of each row
        _, predicted = torch.max(outputs,1)
        total+=labels.size(0)
        # number of correctly predicted class is increased only when the predicted class is equal to the actual class
        # item() is used to convert the tensor to python integer
        correct+=(predicted==labels).sum().item()
    accuracy=100 * correct/total
    print(f"Test Accuracy: {accuracy:.2f}%")

    


Test Accuracy: 77.28%


In [37]:
# save the model
torch.save(model.state_dict(), "cnn_cifar10.pth")

## 4.Identify Overfitting from Training Logs and Solve It
### Problem: You notice the training accuracy increases but validation accuracy stagnates. Modify the model using dropout and early stopping. (use mnist dataset)


In [1]:
# importing the libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

In [2]:
# Load the dataset
# Mentioning the preprocessing steps
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,),(0.5,))
])

train_dataset = torchvision.datasets.MNIST(
    root='/data',
    train=True,
    download=True,
    transform=transform
    )
test_dataset = torchvision.datasets.MNIST(
    root='/data',
    train=False,
    download=True,
    transform=transform
)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=True)


In [3]:
# Building the CNN model with dropout
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.features=nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding =1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier=nn.Sequential(
            nn.Flatten(),

            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),

            #To reduce overfitting added dropout
            nn.Dropout(0.5),

            nn.Linear(128,10)
        )

    def forward(self,x):
        x=self.features(x)
        x=self.classifier(x)
        return x

## dropout(0.5) is nothing but randomly switching off 50 % of neurons in training phase


In [4]:
# model creation
model = CNN()

In [5]:
# Defining loss function and optimizer
criterion=nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [46]:
# Early stopping Variable 
# stores lowest validation loss
best_val_loss=float("inf")
# wait for 3 epochs without improvement
patience=3
# Counts consecutive epochs without improvement
counter=0

epochs = 20

for epoch in range(epochs):

#Training the model
    model.train()
# train_total:total images in train folder
# train_correct: number of training images predicted correctly
# running_loss : total loss during training period    
    train_correct = 0
    train_total = 0
    running_loss = 0

    for images, labels in train_loader:

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        train_total += labels.size(0)

        train_correct += (predicted == labels).sum().item()

    train_accuracy = 100 * train_correct / train_total

# Validating the model
    model.eval()

    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)

            val_correct += (predicted == labels).sum().item()

    val_loss /= len(test_loader)

    val_accuracy = 100 * val_correct / val_total

    print(f"Epoch {epoch+1}")
    print(f"Training Accuracy   : {train_accuracy:.2f}%")
    print(f"Validation Accuracy : {val_accuracy:.2f}%")
    print(f"Validation Loss     : {val_loss:.4f}")

#Implementing early stopping
# checking if current val_loss is less than the best_val_loss( if the loss decreases than the best val loss then the counter is incremented)
# if the loss is not improved for 3 consecutive epochs then the early stopping is triggered    
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        counter = 0

    else:

        counter += 1

    if counter >= patience:

        print("Early stopping triggered!")

        break

Epoch 1
Training Accuracy   : 99.77%
Validation Accuracy : 99.34%
Validation Loss     : 0.0462
Epoch 2
Training Accuracy   : 99.73%
Validation Accuracy : 99.42%
Validation Loss     : 0.0468
Epoch 3
Training Accuracy   : 99.74%
Validation Accuracy : 99.35%
Validation Loss     : 0.0372
Epoch 4
Training Accuracy   : 99.77%
Validation Accuracy : 99.25%
Validation Loss     : 0.0491
Epoch 5
Training Accuracy   : 99.78%
Validation Accuracy : 99.30%
Validation Loss     : 0.0491
Epoch 6
Training Accuracy   : 99.77%
Validation Accuracy : 99.32%
Validation Loss     : 0.0400
Early stopping triggered!


## 5: Transfer Learning with Pretrained VGG16 (Cats vs Dogs)
### Problem: Use VGG16 for binary classification with fine-tuning


In [1]:
#Import the libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision 
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torchvision import datasets
from torch.utils.data import DataLoader,random_split
from torchvision.models import vgg16, VGG16_Weights

In [2]:
# Image Preprocessing
transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456,0.406],
        std=[0.229,0.224,0.225]
    )
]) 

In [3]:
# Load the dataset
dataset=ImageFolder(
    root="PetImages",
    transform=transform
)

In [4]:
# Class verification
print(dataset.classes)
print(dataset.class_to_idx)

['Cat', 'Dog']
{'Cat': 0, 'Dog': 1}


In [5]:
# split the dataset into train and val folder
train_size=int(0.8*len(dataset))
val_size=len(dataset)-train_size

train_dataset,val_dataset=random_split(
    dataset,
    [train_size,val_size]
)


In [6]:
# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

In [7]:
# Load the pre-trained model VGG16
model=vgg16(weights=VGG16_Weights.DEFAULT)

In [8]:
# Freeze the Feature extraction layers
for param in model.features.parameters():
    param.requires_grad=False

In [9]:
# Replace the final layer 
# Original VGG16 can predicts 1000 classes, here we need only cat and dog so we mention the out_features as 2
model.classifier[6]=nn.Linear(
    in_features=4096,
    out_features=2
)

In [10]:
# Fine tuning
# unfreeze the convolution block
for param in model.features[24:].parameters():
    param.requires_grad=True

In [11]:
# Loss function 
criterion=nn.CrossEntropyLoss()

In [12]:
# Optimizer
optimizer=optim.Adam(
    filter(lambda p: p.requires_grad,
           model.parameters()),
    lr=0.0001
)

In [13]:
# Training the model
epochs=2
for epoch in range(epochs):
    model.train()
    correct=0
    total=0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs=model(images)
        loss=criterion(outputs,labels)
        loss.backward()
        optimizer.step()
        _, predicted = torch.max(outputs,1)
        total+=labels.size(0)
        correct+=(predicted==labels).sum().item()
    train_accuracy=100* correct/total
    print(f"Epoch{epoch+1}")
    print(f"Training Accuracy:{train_accuracy:.2f}%")
        


C:\Users\Inst_\anaconda3\envs\aiml1\Lib\site-packages\PIL\TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Epoch1
Training Accuracy:98.20%
Epoch2
Training Accuracy:99.28%


In [14]:
# Validation 
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in val_loader:

        outputs = model(images)

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

val_accuracy = 100 * correct / total

print("Validation Accuracy:", val_accuracy)

Validation Accuracy: 98.82
